In [1]:
import os,shutil

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
base_dir='/content/drive/MyDrive/C-NMC_Leukemia'

In [4]:
train_dir=os.path.join(base_dir,'training_data')

In [5]:
validation_dir=os.path.join(base_dir,'validation_data')

In [6]:
test_dir=os.path.join(base_dir,'testing_data')

In [7]:
from keras import layers
from keras import models

In [8]:
model=models.Sequential()
model.add(layers.Conv2D(32,(3,3),activation='relu',input_shape=(150,150,3)))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(64,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(128,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))
model.add(layers.Conv2D(128,(3,3),activation='relu'))
model.add(layers.MaxPooling2D((2,2)))


In [9]:
model.add(layers.Flatten())
model.add(layers.Dense(512,activation='relu'))
model.add(layers.Dense(1,activation='sigmoid'))

In [10]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 148, 148, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 74, 74, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 72, 72, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 36, 36, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 34, 34, 128)       73856     
                                                                 
 max_pooling2d_2 (MaxPoolin  (None, 17, 17, 128)       0

In [11]:
from tensorflow.keras import optimizers
model.compile(loss='binary_crossentropy',
             optimizer=optimizers.RMSprop(learning_rate=1e-4),
             metrics=['accuracy'])

In [12]:
from keras.preprocessing.image import ImageDataGenerator

In [13]:
train_datagen=ImageDataGenerator(rescale=1/.255)
test_datagen=ImageDataGenerator(rescale=1/.255)

In [14]:
trainingData_generator=train_datagen.flow_from_directory(train_dir,
                                                        target_size=(150,150),
                                                        class_mode='binary')

Found 10661 images belonging to 3 classes.


In [15]:
validationData_generator=test_datagen.flow_from_directory(validation_dir,
                                                        target_size=(150,150),
                                                        class_mode='binary')

Found 1867 images belonging to 1 classes.


In [16]:
def generator():
    i=0
    while True:
        i+=1
        yield i

In [17]:
for data_batch,labels_batch in trainingData_generator:
    print("Shape of data batch:", data_batch.shape)
    print("Shape of labels batch:", labels_batch.shape)
    break

Shape of data batch: (32, 150, 150, 3)
Shape of labels batch: (32,)


In [19]:
history=model.fit(trainingData_generator,
                           epochs=10,
                           validation_data=validationData_generator)

Epoch 1/10
334/334 [==============================] - 636s 2s/step - loss: -1276593408.0000 - accuracy: 0.3359 - val_loss: 49648652288.0000 - val_accuracy: 0.0000e+00
Epoch 2/10
334/334 [==============================] - 627s 2s/step - loss: -2535696640.0000 - accuracy: 0.3359 - val_loss: 98832949248.0000 - val_accuracy: 0.0000e+00
Epoch 3/10
334/334 [==============================] - 632s 2s/step - loss: -4696399360.0000 - accuracy: 0.3359 - val_loss: 171802935296.0000 - val_accuracy: 0.0000e+00
Epoch 4/10
334/334 [==============================] - 631s 2s/step - loss: -8104651264.0000 - accuracy: 0.3359 - val_loss: 300947079168.0000 - val_accuracy: 0.0000e+00
Epoch 5/10
334/334 [==============================] - 633s 2s/step - loss: -14303305728.0000 - accuracy: 0.3359 - val_loss: 507359821824.0000 - val_accuracy: 0.0000e+00
Epoch 6/10
334/334 [==============================] - 628s 2s/step - loss: -23073394688.0000 - accuracy: 0.3359 - val_loss: 795130462208.0000 - val_accuracy: 0.0

In [20]:
model.save('leukemia_detection')

In [21]:
from keras.applications.vgg16 import VGG16

In [22]:
conv_base=VGG16(weights='imagenet',
               include_top=False,
               input_shape=(150,150,3))

58889256/58889256 [==============================] - 0s 0us/step


In [23]:
conv_base.summary()

Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 150, 150, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 150, 150, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 150, 150, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 75, 75, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 75, 75, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 75, 75, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 37, 37, 128)       0     

In [24]:
import numpy as np
from keras.preprocessing.image import ImageDataGenerator

In [25]:
datagen=ImageDataGenerator(rescale=1/.255)

In [45]:
def extract_features(directory,sample_count):
    features=np.zeros(shape=(sample_count,4,4,512))
    labels=np.zeros(shape=(sample_count))
    generator=datagen.flow_from_directory(
        directory,
        target_size=(150,150),
        batch_size=20,
        class_mode='binary')
    i=0
    for inputs_batch,labels_batch in generator:
        features_batch=conv_base.predict(inputs_batch)
       # features_batch = features_batch[:15]
        features[i*20:(i+1)*20]=features_batch
        #features[:15]=features_batch
        #features = np.zeros((20, 4, 4, 512))
        ##features_batch = features_batch[:15]
        #labels = labels.reshape((20, 1))
        #labels = np.concatenate((labels, labels_batch), axis=1)
        labels[i*20:(i+1)*20]=labels_batch
        i+=1
        if i*20>=sample_count:
            break

        return features,labels

In [46]:
train_features,train_labels=extract_features(train_dir,10661)
validation_features,validation_labels=extract_features(validation_dir,1867)
test_features,test_labels=extract_features(test_dir,624)

Found 10661 images belonging to 3 classes.
1/1 [==============================] - 7s 7s/step
Found 1867 images belonging to 1 classes.
1/1 [==============================] - 5s 5s/step
Found 2586 images belonging to 1 classes.
1/1 [==============================] - 5s 5s/step


In [48]:
train_features=np.reshape(train_features,(10661,4*4*512))
validation_features=np.reshape(validation_features,(1867,4*4*512))
test_features=np.reshape(test_features,(624,4*4*512))

In [49]:
model=models.Sequential()
model.add(layers.Dense(256,activation='relu',input_dim=4*4*512))
model.add(layers.Dropout(0,5))
model.add(layers.Dense(1,activation='sigmoid'))

In [52]:
from tensorflow.keras import optimizers
model.compile(optimizer=optimizers.RMSprop(learning_rate=2e-5),
             loss='binary_crossentropy',
             metrics=['acc'])

In [53]:
history=model.fit(train_features,train_labels,
                 epochs=20,
                 batch_size=20,
                 validation_data=(validation_features,validation_labels))

Epoch 1/20
534/534 [==============================] - 19s 35ms/step - loss: 0.6947 - acc: 0.9989 - val_loss: 0.7803 - val_acc: 0.9914
Epoch 2/20
534/534 [==============================] - 11s 21ms/step - loss: 0.6686 - acc: 0.9992 - val_loss: 0.9568 - val_acc: 0.9893
Epoch 3/20
534/534 [==============================] - 12s 23ms/step - loss: 0.6390 - acc: 0.9992 - val_loss: 1.0607 - val_acc: 0.9893
Epoch 4/20
534/534 [==============================] - 13s 25ms/step - loss: 0.6131 - acc: 0.9990 - val_loss: 0.9709 - val_acc: 0.9893
Epoch 5/20
534/534 [==============================] - 12s 23ms/step - loss: 0.5800 - acc: 0.9992 - val_loss: 1.0622 - val_acc: 0.9893
Epoch 6/20
534/534 [==============================] - 12s 22ms/step - loss: 0.5480 - acc: 0.9990 - val_loss: 0.9366 - val_acc: 0.9893
Epoch 7/20
534/534 [==============================] - 13s 23ms/step - loss: 0.5126 - acc: 0.9990 - val_loss: 0.8557 - val_acc: 0.9893
Epoch 8/20
534/534 [==============================] - 12s 23ms